# Evaluation — Head-to-Head and Round-Robin

Interactive tool for comparing Connect-4 agents. Pick two or more models, set the number of
games, and hit Run. The library code lives in `src/eval.py`; this notebook is just the
control panel.

**Selection rules:**
- **2 models** -> single head-to-head match (alternating first-player)
- **3 or more models** -> round-robin (every pair plays every other pair)
- **0 or 1 selected** -> error

All `ModelAgent`s below use **greedy** move selection (argmax) with **tactical overrides
on** (immediate-win and immediate-block) — mirroring how the agent will be deployed for
the tournament. Instantiate a copy with `use_tactics=False` if you want the pure
model-vs-model comparison for analysis.

In [ ]:
# ── Cell 1: Imports and model loading ────────────────────────────────────────
import os, sys

# ── Repo coordinates ─────────────────────────────────────────────────────────
# If the repo is ever renamed, forked, or you want to test a branch, change
# these values. Every git / path lookup in this notebook uses them.
GITHUB_OWNER    = "Stiles-Clements1"
GITHUB_REPO     = "connect4-rl-arena"
GITHUB_BRANCH   = "main"
GITHUB_URL      = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"

# Where the repo lives on each platform
COLAB_REPO_PATH = f"/content/{GITHUB_REPO}"
LOCAL_REPO_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    # Opened from the Colab badge on GitHub. Clone the repo into /content
    # (if not already there) so src.model_loader and src.eval are importable.
    if not os.path.isdir(COLAB_REPO_PATH):
        !git clone --branch {GITHUB_BRANCH} {GITHUB_URL} {COLAB_REPO_PATH}
    else:
        # Repo already cloned this runtime — pull the latest commits so a
        # re-run picks up any pushes made since the original clone.
        print(f"Repo already present at {COLAB_REPO_PATH}; pulling latest {GITHUB_BRANCH}...")
        !cd {COLAB_REPO_PATH} && git fetch --quiet origin && git checkout --quiet {GITHUB_BRANCH} && git pull --quiet origin {GITHUB_BRANCH}
    %cd {COLAB_REPO_PATH}
    !pip install -q tqdm ipywidgets
    REPO_ROOT = COLAB_REPO_PATH
else:
    # Local Jupyter / Cursor run — assume the notebook lives in notebooks/
    REPO_ROOT = LOCAL_REPO_PATH

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Purge any previously-imported `src.*` modules so a re-run picks up code
# that was freshly pulled / re-cloned this session. Without this, Python
# keeps the old modules in sys.modules and re-imports are a no-op.
for _m in [k for k in sys.modules if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]

print(f"Running in {'Colab' if IS_COLAB else 'local'} mode.")
print(f"REPO_ROOT = {REPO_ROOT}")
print(f"GitHub    = {GITHUB_URL} @ {GITHUB_BRANCH}")

# ── Hardware init — MUST happen BEFORE importing model_loader ───────────────
# Keras models are placed on whichever device TF sees first. If TF is imported
# and queried only AFTER load_all_models(), each model gets pinned to CPU
# even when an A100 is available, and nothing later can move them. Doing the
# detection here ensures models land on the GPU from the start.
import tensorflow as tf
_gpus = tf.config.list_physical_devices("GPU")
for _g in _gpus:
    tf.config.experimental.set_memory_growth(_g, True)
HARDWARE = f"GPU ({_gpus[0].name.split(':')[-1]})" if _gpus else "CPU only"
print(f"Hardware  = {HARDWARE}")

from src import model_loader
from src.eval import (
    ModelAgent, RandomAgent, MinimaxAgent,
    play_match, play_match_parallel, run_round_robin,
    format_result, round_robin_to_dataframe,
    save_results_json, save_win_rate_heatmap,
)

models = model_loader.load_all_models()
print(f"\nLoaded {len(models)} models.")

In [ ]:
# ── Cell 2: Wrap each model as an Agent ─────────────────────────────────────
# HARDWARE was set in Cell 1 (before model_loader imported, so TF has the GPU
# initialised in time to place the models there). Evaluation always uses the
# batched (parallel-games) path: one forward pass per half-move per agent,
# covering all concurrent games. Dramatic speedup on GPU, modest on CPU.
print(f"Hardware in use: {HARDWARE}\n")


# Defaults: greedy=True (argmax), use_tactics=True (immediate win / immediate block).
# These are the settings the deployed tournament agent will use.

agents = {
    name: ModelAgent(wrapper, greedy=True, use_tactics=True)
    for name, wrapper in models.items()
}
agents["random"]     = RandomAgent()

# Minimax baselines at multiple depths — calibrated non-neural opponents.
# Depth 1 is barely tactical; depth 3 sees short threats; depth 5 is strong.
# Add more depths here (e.g. 7) if you want, but depth 7+ takes hundreds of
# ms per move on CPU.
agents["minimax_d1"] = MinimaxAgent(depth=1)
agents["minimax_d3"] = MinimaxAgent(depth=3)
agents["minimax_d5"] = MinimaxAgent(depth=5)

print(f"Built {len(agents)} agents:")
for name in agents:
    print(f"  - {name}")

In [ ]:
# ── Cell 3: Interactive evaluation UI ────────────────────────────────────────
import ipywidgets as widgets
from IPython.display import display, clear_output

# One checkbox per agent
checkboxes = {
    name: widgets.Checkbox(value=False, description=name, indent=False)
    for name in agents
}

n_games_slider = widgets.IntSlider(
    value=100, min=10, max=500, step=10,
    description='N games per pair:',
    style={'description_width': 'initial'},
    continuous_update=False,
)

run_button = widgets.Button(
    description='Run evaluation',
    button_style='primary',
    icon='play',
    layout=widgets.Layout(width='200px'),
)

output = widgets.Output()

def on_run_clicked(b):
    with output:
        clear_output()
        selected = [name for name, cb in checkboxes.items() if cb.value]
        n = len(selected)

        if n < 2:
            print(f"ERROR: select at least 2 models ({n} currently selected).")
            return

        n_games = n_games_slider.value

        if n == 2:
            a, b_ = selected
            print(f"Head-to-head: {a} vs {b_} ({n_games} games, alternating first, {HARDWARE})\n")
            result = play_match_parallel(agents[a], agents[b_], n_games=n_games)
            print(format_result(result))

            # Auto-save the raw match result to logs/ so we do not lose it
            # when the Colab runtime disconnects. The tag goes into the
            # filename so multiple runs do not overwrite each other.
            tag  = f"{a}_vs_{b_}_{n_games}g"
            meta = {"hardware": HARDWARE, "n_games": n_games, "mode": "head_to_head"}
            json_path = save_results_json(result, tag=tag, metadata=meta)
            print(f"\nSaved:  {json_path}")
        else:
            print(f"Round-robin: {n} agents, {n_games} games per pair ({HARDWARE})\n")
            subset = {k: agents[k] for k in selected}
            results = run_round_robin(subset, n_games=n_games, parallel=True)
            df = round_robin_to_dataframe(results, selected)

            print("Win-rate matrix — row agent's win rate when playing column agent:")
            print((df * 100).round(1).fillna('--').to_string())
            print()

            # Overall ranking by mean win rate across the other agents
            ranking = df.mean(axis=1).sort_values(ascending=False)
            print("Overall ranking (mean win rate across all opponents):")
            for i, (name, wr) in enumerate(ranking.items(), 1):
                print(f"  {i}. {name:30s}  {wr:.1%}")

            # Auto-save: raw JSON to logs/, heatmap PNG to report/figures/.
            # The PNG is sized and styled for pasting straight into the report.
            tag  = f"roundrobin_{n}agents_{n_games}g"
            meta = {"hardware": HARDWARE, "n_games": n_games, "mode": "round_robin",
                    "agents": selected}
            json_path = save_results_json(results, tag=tag, metadata=meta)
            png_path  = save_win_rate_heatmap(
                df,
                title=f"Round-robin win rates ({n_games} games/pair, {n} agents)",
                tag=tag,
            )
            print(f"\nSaved:  {json_path}")
            print(f"Heatmap: {png_path}")

run_button.on_click(on_run_clicked)

ui = widgets.VBox([
    widgets.HTML("<h3>Select 2 or more models</h3>"),
    widgets.VBox(list(checkboxes.values())),
    widgets.HTML("<br>"),
    n_games_slider,
    run_button,
    widgets.HTML("<hr>"),
    output,
])
display(ui)

In [ ]:
# ── Cell 4 (optional): Programmatic use, no UI ───────────────────────────────
# Useful for scripting a specific matchup or saving results to disk.

# Example: quick M1 vs every baseline
# results = {}
# opponents = ["random", "stiles_cnn", "luke_cnn", "zan_transformer"]
# for opp in opponents:
#     r = play_match(agents["m1"], agents[opp], n_games=100)
#     results[opp] = r
#     print(format_result(r))
#     print()